In [1]:
import datetime
from functools import partial

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

import torch
from torch import nn
from torch import optim
from torch.utils.data import DataLoader, TensorDataset

from moabb.datasets import BNCI2014_001


from tools.dataloading import get_raw_data, create_dataloaders
from tools.preprocessing import apply_mne_ica
from tools.training import Trainer

In [2]:
# 1. Pipeline Configuration
dataset = BNCI2014_001()

# Setting up standard industry parameters for testing a baseline
hyperparams = {
    "dataset": type(dataset).__name__,
    "split_mode": "cross",
    "test_subject_id": 1,
    "batch_size": 32,
    "sample_frequency": 128,
    "learning_rate": 0.001,
    "epochs": 500,
    "patience": 50,
    "dropout_rate": 0.35,
    "n_classes": 4,
    "electrode_channels": 22,
    "preprocessing": "ICA"
}

# Add dynamic timestamp string to organize logs
timestamp = datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
log_dir = f"models/eegnet/results/baseline_{hyperparams['dataset']}_{hyperparams['split_mode']}_{timestamp}"

In [3]:
X, y, metadata = get_raw_data(
    dataset, n_subjects=-1, sample_frequency=hyperparams["sample_frequency"])

# Extract dynamic dimensions from the dataset that the model needs
hyperparams["n_classes"] = metadata["n_classes"]
hyperparams["electrode_channels"] = metadata["electrode_channels"]
hyperparams["sample_length"] = metadata["sample_length"]


X, y, metadata = apply_mne_ica(
    X, y, metadata, sfreq=hyperparams["sample_frequency"], n_components=15)

Choosing from all possible events


Applying MNE ICA (method=fastica, n_components=15) to epoched data...
Fitting ICA to data using 22 channels (please be patient, this may take a while)
Selecting by number: 15 components
Fitting ICA took 39.2s.
Applying ICA to Epochs instance
    Transforming to ICA space (15 components)
    Zeroing out 0 ICA components
    Projecting back using 22 PCA components


In [4]:
# Reshape to (samples, channels, time) for PyTorch
X = X.reshape(X.shape[0], X.shape[2], X.shape[1])

In [5]:
train_loader, test_loader, metadata = create_dataloaders(
    X, y, metadata, split_mode=hyperparams["split_mode"], test_subject_id=hyperparams["test_subject_id"], batch_size=hyperparams["batch_size"])



Split Mode: CROSS | test_size=0.2 | random_state=42
Training on 4608 trials
Testing on 576 trials
Input Shape for Model: torch.Size([32, 1, 513, 22])
Batch size: 32 | Nº Channels: 513 | Sample length: 22


### Model Configuration

In [6]:
class SpatioTemporalBranch(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, n_electrodes, dropout_rate):
        super(SpatioTemporalBranch, self).__init__()

        # Temporal (Yellow) - Set bias=False, removed intermediate dropout/activation
        self.temporal = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=(kernel_size, 1),
                      padding="same", bias=False)
        )

        # Spatial Depthwise (Orange) - bias=False, combined BatchNorm and Dropout
        self.spatial = nn.Sequential(
            nn.Conv2d(out_channels, out_channels, kernel_size=(1, n_electrodes),
                      groups=out_channels, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ELU(),
            nn.Dropout(dropout_rate)
        )

    def forward(self, x):
        x = self.temporal(x)
        x = self.spatial(x)
        return x


class SepConv2d(nn.Module):
    """Helper module for Depthwise Separable Convolutions to prevent parameter inflation"""

    def __init__(self, in_channels, out_channels, kernel_size, padding="same"):
        super(SepConv2d, self).__init__()
        self.depthwise = nn.Conv2d(in_channels, in_channels, kernel_size=kernel_size,
                                   groups=in_channels, padding=padding, bias=False)
        self.pointwise = nn.Conv2d(
            in_channels, out_channels, kernel_size=(1, 1), bias=False)

    def forward(self, x):
        return self.pointwise(self.depthwise(x))


class EEGInception(nn.Module):
    def __init__(self, n_electrodes, n_classes, dropout_rate=0.35):
        super(EEGInception, self).__init__()

        # MODULE 1: 3 branches, each 8 filters. Total output = 24 filters.
        self.branch1_1 = SpatioTemporalBranch(
            1, 8, 64, n_electrodes, dropout_rate)
        self.branch1_2 = SpatioTemporalBranch(
            1, 8, 32, n_electrodes, dropout_rate)
        self.branch1_3 = SpatioTemporalBranch(
            1, 8, 16, n_electrodes, dropout_rate)

        # A1: Pooling
        self.pool1 = nn.AvgPool2d(kernel_size=(4, 1))

        # MODULE 2: Replaced standard Conv2d with Depthwise Separable (SepConv2d)
        self.branch2_1 = nn.Sequential(
            SepConv2d(24, 8, kernel_size=(16, 1), padding="same"),
            nn.BatchNorm2d(8),
            nn.ELU(),
            nn.Dropout(dropout_rate)
        )
        self.branch2_2 = nn.Sequential(
            SepConv2d(24, 8, kernel_size=(8, 1), padding="same"),
            nn.BatchNorm2d(8),
            nn.ELU(),
            nn.Dropout(dropout_rate)
        )
        self.branch2_3 = nn.Sequential(
            SepConv2d(24, 8, kernel_size=(4, 1), padding="same"),
            nn.BatchNorm2d(8),
            nn.ELU(),
            nn.Dropout(dropout_rate)
        )

        # A2: Pooling
        self.pool2 = nn.AvgPool2d(kernel_size=(2, 1))

        # OUTPUT MODULE: Replaced standard Conv2d with Depthwise Separable
        self.out_conv1 = nn.Sequential(
            SepConv2d(24, 12, kernel_size=(8, 1), padding="same"),
            nn.BatchNorm2d(12),
            nn.ELU(),
            nn.Dropout(dropout_rate),
            nn.AvgPool2d(kernel_size=(2, 1))  # A3
        )
        self.out_conv2 = nn.Sequential(
            SepConv2d(12, 6, kernel_size=(4, 1), padding="same"),
            nn.BatchNorm2d(6),
            nn.ELU(),
            nn.Dropout(dropout_rate),
            nn.AvgPool2d(kernel_size=(2, 1))  # A4
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.LazyLinear(n_classes)
        )

    def apply_max_norm(self, max_norm_value=1.0):
        """EEG regularizer to constraint the weights of the classification layer"""
        with torch.no_grad():
            for module in self.classifier.modules():
                if hasattr(module, 'weight') and module.weight is not None:
                    norm = module.weight.norm(2, dim=0, keepdim=True)
                    desired = torch.clamp(norm, 0, max_norm_value)
                    module.weight *= (desired / (norm + 1e-8))

    def forward(self, x, **kwargs):
        # x shape: (Batch, 1, Time, Electrodes)

        # Module 1
        b1 = torch.cat([self.branch1_1(x), self.branch1_2(x),
                       self.branch1_3(x)], dim=1)
        x = self.pool1(b1)

        # Module 2
        b2 = torch.cat([self.branch2_1(x), self.branch2_2(x),
                       self.branch2_3(x)], dim=1)
        x = self.pool2(b2)

        # Output Module
        x = self.out_conv1(x)
        x = self.out_conv2(x)

        return self.classifier(x)

In [7]:
# 3. Model Initialization
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

model = EEGInception(
    n_classes=hyperparams["n_classes"],
    n_electrodes=hyperparams["electrode_channels"]
).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=hyperparams["learning_rate"])

Using device: cuda


In [8]:
print(model)

EEGInception(
  (branch1_1): SpatioTemporalBranch(
    (temporal): Sequential(
      (0): Conv2d(1, 8, kernel_size=(64, 1), stride=(1, 1), padding=same, bias=False)
    )
    (spatial): Sequential(
      (0): Conv2d(8, 8, kernel_size=(1, 22), stride=(1, 1), groups=8, bias=False)
      (1): BatchNorm2d(8, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ELU(alpha=1.0)
      (3): Dropout(p=0.35, inplace=False)
    )
  )
  (branch1_2): SpatioTemporalBranch(
    (temporal): Sequential(
      (0): Conv2d(1, 8, kernel_size=(32, 1), stride=(1, 1), padding=same, bias=False)
    )
    (spatial): Sequential(
      (0): Conv2d(8, 8, kernel_size=(1, 22), stride=(1, 1), groups=8, bias=False)
      (1): BatchNorm2d(8, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ELU(alpha=1.0)
      (3): Dropout(p=0.35, inplace=False)
    )
  )
  (branch1_3): SpatioTemporalBranch(
    (temporal): Sequential(
      (0): Conv2d(1, 8, kernel_size=(16, 1), stride=(

In [9]:
# 4. Training and Evaluation Tracking
trainer = Trainer(
    model=model,
    optimizer=optimizer,
    criterion=criterion,
    device=device,
    log_dir=log_dir,
    experiment_config=hyperparams
)

trainer.train(
    train_loader=train_loader,
    test_loader=test_loader,
    epochs=hyperparams["epochs"],
    patience=hyperparams["patience"]
)

Starting training on cuda...
Logging TensorBoard to: models/eegnet/results/baseline_BNCI2014_001_cross_20260509-230641


/home/carlos/.conda/envs/ml/lib/python3.13/site-packages/torch/nn/modules/conv.py:543: UserWarning: Using padding='same' with even kernel lengths and odd dilation may require a zero-padded copy of the input be created (Triggered internally at /pytorch/aten/src/ATen/native/Convolution.cpp:1027.)
  return F.conv2d(


Epoch [1/500] | Train Loss: 1.4192 | Test Loss: 1.3816 | Test Acc: 28.30%
Epoch [10/500] | Train Loss: 1.3501 | Test Loss: 1.3896 | Test Acc: 27.60%
Epoch [20/500] | Train Loss: 1.3217 | Test Loss: 1.3961 | Test Acc: 27.43%
Epoch [30/500] | Train Loss: 1.2979 | Test Loss: 1.3948 | Test Acc: 29.34%
Epoch [40/500] | Train Loss: 1.2824 | Test Loss: 1.3926 | Test Acc: 30.38%
Epoch [50/500] | Train Loss: 1.2770 | Test Loss: 1.3916 | Test Acc: 31.42%
Early stopping triggered after 56 epochs.
Training complete. Best model saved to: models/eegnet/results/baseline_BNCI2014_001_cross_20260509-230641/best_model.pth
